In [1]:
import pandas as pd
import sqlite3
from pathlib import Path

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
# Load CRM dataset
df = pd.read_csv("crm_data.csv")

# Display dataset information
print("CRM Dataset Loaded Successfully!")
print("Dataset Shape:", df.shape)

# Display first 5 records
display(df.head())

CRM Dataset Loaded Successfully!
Dataset Shape: (100, 10)


,customer_id,customer_name,company,lead_source,sales_rep,deal_value,deal_stage,status,created_date,last_activity
0,CUST0001,Rohan Reddy,Keystone Digital,Trade Show,Priya Shah,155000,New Lead,Active,2026-10-17,2026-10-20
1,CUST0002,Kiran Joshi,Aster Technologies,Cold Call,Rahul Menon,160000,New Lead,Active,2025-03-31,2026-06-06
2,CUST0003,Aarav Nair,Lumina Healthcare,Email Campaign,Sneha Iyer,190000,Qualified,Active,2026-10-28,2026-11-14
3,CUST0004,Ishaan Menon,BluePeak Solutions,LinkedIn,Sneha Iyer,80000,Qualified,Inactive,2025-10-12,2025-12-30
4,CUST0005,Meera Menon,BluePeak Solutions,Cold Call,Sneha Iyer,355000,Won,Inactive,2026-09-11,2026-10-14


In [3]:
# Dataset structure
print("Dataset Information:")
df.info()

print("\nMissing Values:")
display(df.isnull().sum())

print("\nDuplicate Records:", df.duplicated().sum())

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   customer_id    100 non-null    object
 1   customer_name  100 non-null    object
 2   company        100 non-null    object
 3   lead_source    100 non-null    object
 4   sales_rep      100 non-null    object
 5   deal_value     100 non-null    int64 
 6   deal_stage     100 non-null    object
 7   status         100 non-null    object
 8   created_date   100 non-null    object
 9   last_activity  100 non-null    object
dtypes: int64(1), object(9)
memory usage: 7.9+ KB

Missing Values:


customer_id      0
customer_name    0
company          0
lead_source      0
sales_rep        0
deal_value       0
deal_stage       0
status           0
created_date     0
last_activity    0
dtype: int64


Duplicate Records: 0


In [4]:
# Convert date columns to datetime
df["created_date"] = pd.to_datetime(df["created_date"])
df["last_activity"] = pd.to_datetime(df["last_activity"])

# Verify data types
print("Date Conversion Completed!")
print(df[["created_date", "last_activity"]].dtypes)

# Display updated dataset
display(df.head())


Date Conversion Completed!
created_date     datetime64[ns]
last_activity    datetime64[ns]
dtype: object


,customer_id,customer_name,company,lead_source,sales_rep,deal_value,deal_stage,status,created_date,last_activity
0,CUST0001,Rohan Reddy,Keystone Digital,Trade Show,Priya Shah,155000,New Lead,Active,2026-10-17,2026-10-20
1,CUST0002,Kiran Joshi,Aster Technologies,Cold Call,Rahul Menon,160000,New Lead,Active,2025-03-31,2026-06-06
2,CUST0003,Aarav Nair,Lumina Healthcare,Email Campaign,Sneha Iyer,190000,Qualified,Active,2026-10-28,2026-11-14
3,CUST0004,Ishaan Menon,BluePeak Solutions,LinkedIn,Sneha Iyer,80000,Qualified,Inactive,2025-10-12,2025-12-30
4,CUST0005,Meera Menon,BluePeak Solutions,Cold Call,Sneha Iyer,355000,Won,Inactive,2026-09-11,2026-10-14


In [5]:
# Create SQLite database
db_path = "crm_database.db"

conn = sqlite3.connect(db_path)

# Store DataFrame as SQLite table
df.to_sql("customers", conn, if_exists="replace", index=False)

print("CRM Database Created Successfully!")

# Verify table
tables = pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table'",
    conn
)

display(tables)

CRM Database Created Successfully!


,name
0,customers


In [6]:
# Verify total records
record_count = pd.read_sql(
    "SELECT COUNT(*) AS total_records FROM customers",
    conn
)

print("Total CRM Records:")
display(record_count)

# Preview database records
print("Database Preview:")
display(pd.read_sql("SELECT * FROM customers LIMIT 5", conn))

Total CRM Records:


,total_records
0,100


Database Preview:


,customer_id,customer_name,company,lead_source,sales_rep,deal_value,deal_stage,status,created_date,last_activity
0,CUST0001,Rohan Reddy,Keystone Digital,Trade Show,Priya Shah,155000,New Lead,Active,2026-10-17 00:00:00,2026-10-20 00:00:00
1,CUST0002,Kiran Joshi,Aster Technologies,Cold Call,Rahul Menon,160000,New Lead,Active,2025-03-31 00:00:00,2026-06-06 00:00:00
2,CUST0003,Aarav Nair,Lumina Healthcare,Email Campaign,Sneha Iyer,190000,Qualified,Active,2026-10-28 00:00:00,2026-11-14 00:00:00
3,CUST0004,Ishaan Menon,BluePeak Solutions,LinkedIn,Sneha Iyer,80000,Qualified,Inactive,2025-10-12 00:00:00,2025-12-30 00:00:00
4,CUST0005,Meera Menon,BluePeak Solutions,Cold Call,Sneha Iyer,355000,Won,Inactive,2026-09-11 00:00:00,2026-10-14 00:00:00


In [7]:
# CRM SQL Analysis

queries = {
    "Total Customers": """
        SELECT COUNT(*) AS total_customers
        FROM customers
    """,

    "Total Deal Value": """
        SELECT SUM(deal_value) AS total_deal_value
        FROM customers
    """,

    "Deal Stage Distribution": """
        SELECT deal_stage, COUNT(*) AS customer_count
        FROM customers
        GROUP BY deal_stage
        ORDER BY customer_count DESC
    """,

    "Lead Source Distribution": """
        SELECT lead_source, COUNT(*) AS total_leads
        FROM customers
        GROUP BY lead_source
        ORDER BY total_leads DESC
    """
}

for title, query in queries.items():
    print(f"\n--- {title} ---")
    display(pd.read_sql_query(query, conn))


--- Total Customers ---


,total_customers
0,100



--- Total Deal Value ---


,total_deal_value
0,24875000



--- Deal Stage Distribution ---


,deal_stage,customer_count
0,Qualified,28
1,Won,22
2,Proposal,15
3,New Lead,14
4,Negotiation,11
5,Lost,10



--- Lead Source Distribution ---


,lead_source,total_leads
0,Referral,24
1,LinkedIn,19
2,Email Campaign,19
3,Trade Show,17
4,Cold Call,11
5,Website,10
